# Annotations with various approaches

Annotate harmonized feature tables using multiple similarity approaches: Spectral, Cosine and DreaMS.

In [1]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np

from ms_entropy import clean_spectrum
from benchmarking.utils import to_typed_bank, filter_spectrum_peaks_in_df
from benchmarking.similarity.flashier_entropy import flashier_entropy_search
from benchmarking.similarity.cosine_similarity import flash_cosine_search
import benchmarking.constants as cons

import warnings
from numba.core.errors import NumbaTypeSafetyWarning

# Filter out the specific numba type safety warning
warnings.simplefilter("ignore", category=NumbaTypeSafetyWarning)

tqdm.pandas()

##### Prepare Library

Read in and prepare the spectral library for matching

In [2]:
library_data_filtered = pd.read_parquet(
    "../data/library_spectra/all_sorted_library_spectra.parquet"
)
library_data_filtered.head(2)

,experimental,inchikey_2d,ingest_lib,instrument_type,ion_formula,ionization_mode,normalized_adduct,normalized_intensities,normalized_mzs,num_peaks,precursor_charge,precursor_error_ppm,precursor_mz,NUMBER_FILTERED_PEAKS,esn
721606,True,ZWCXYZRRTRDGQE-UHFFFAOYSA-N,gnps_2025-05-06_2025-06-26,qTof,C99H142N20O17,positive,[M+H2]2+,"[0.0020905364455540906, 0.0028819746066764303,...","[52.005226, 52.967926, 57.472473, 62.060215, 6...",1660.0,2.0,4.697715e+08,2.0,0,None
721607,True,ZWCXYZRRTRDGQE-UHFFFAOYSA-N,gnps_2025-05-06_2025-06-26,qTof,C99H142N20O17,positive,[M+H2]2+,"[0.011945307094692683, 0.00013242163023812931,...","[62.059917, 62.097515, 62.453758, 70.028458, 7...",2023.0,2.0,4.697715e+08,2.0,0,None


In [3]:
# Prepare library for matching
reference_spectra = library_data_filtered.apply(
    lambda x: np.column_stack((x["normalized_mzs"], x["normalized_intensities"])),
    axis=1,
).to_numpy()

reference_spectra = [clean_spectrum(spectrum) for spectrum in reference_spectra]
reference_spectra = np.array(reference_spectra, dtype=object)
reference_spectra_list = to_typed_bank(reference_spectra)


reference_precursors = library_data_filtered["precursor_mz"].to_numpy(dtype=np.float64)
reference_precursors = np.asarray(
    reference_precursors, dtype=np.float64, order="C"
).reshape(-1)

reference_inchis = library_data_filtered["inchikey_2d"].to_numpy(dtype=object)
reference_inchis = np.asarray(reference_inchis, dtype=object, order="C").reshape(-1)

reference_ids = library_data_filtered.index.to_numpy(dtype=object)
reference_ids = np.asarray(reference_ids, dtype=int, order="C").reshape(-1)


reference_mzs = library_data_filtered["normalized_mzs"].to_numpy(dtype=object)
reference_intensities = library_data_filtered["normalized_intensities"].to_numpy(
    dtype=object
)

##### Specify input feature tables

In [4]:
base_dir_public = "../data/public_dataset"
public_datasets = [
    "MSV000090327",
    "MSV000091642",
    "MSV000095813",
    "MSV000097967",
    "MSV000097015",
    "MSV000096291",
    "MSV000096189",
    "ST002402",
    "MSV000084402",
    "MTBLS12332",
]

base_dir_internal = "../data/groundtruth_dataset"
internal_datasets = [
    "MSV000098263",
    "NIST_SRM",
    "plant_spikein",
]

In [5]:
input_feature_tables = []

for dataset in public_datasets:
    input_dir = f"{base_dir_public}/{dataset}/harmonized"
    if not os.path.exists(input_dir):
        print(f"Directory {input_dir} does not exist. Skipping dataset {dataset}.")
        continue

    for file in os.listdir(input_dir):
        input_feature_tables.append(f"{input_dir}/{file}")

for dataset in internal_datasets:
    input_dir = f"{base_dir_internal}/{dataset}/harmonized"
    if not os.path.exists(input_dir):
        print(f"Directory {input_dir} does not exist. Skipping dataset {dataset}.")
        continue

    for file in os.listdir(input_dir):
        input_feature_tables.append(f"{input_dir}/{file}")

In [6]:
output_feature_tables_spectral = []
output_feature_tables_cosine = []


for file_path in input_feature_tables:
    output_feature_tables_spectral.append(
        file_path.replace("harmonized", "annotated_spectral_entropy")
    )
    os.makedirs(os.path.dirname(output_feature_tables_spectral[-1]), exist_ok=True)

    output_feature_tables_cosine.append(
        file_path.replace("harmonized", "annotated_cosine_similarity")
    )
    os.makedirs(os.path.dirname(output_feature_tables_cosine[-1]), exist_ok=True)

# Annotate feature tables
Run annotation on all inputs

## Spectral entropy

In [7]:
counter = 0
for input_feature_table, output_feature_table in tqdm(
    zip(input_feature_tables, output_feature_tables_spectral),
    total=len(input_feature_tables),
):
    if os.path.exists(output_feature_table):
        counter += 1
        continue

    harmonized_feature_table = pd.read_parquet(input_feature_table)
    harmonized_feature_table_spectra_available = harmonized_feature_table[
        harmonized_feature_table["MS/MS_ASSIGNED"] == True
    ]

    harmonized_feature_table_filtered = filter_spectrum_peaks_in_df(
        harmonized_feature_table_spectra_available,
        "MS/MS_INTENSITIES",
        "MS/MS_MZS",
        10.0,
        0.01,
    )

    query_spectra = harmonized_feature_table_filtered.apply(
        lambda x: np.column_stack((x["MS/MS_MZS"], x["MS/MS_INTENSITIES"])), axis=1
    ).to_numpy()

    query_spectra = [clean_spectrum(spectrum) for spectrum in query_spectra]
    query_spectra = np.array(query_spectra, dtype=object)
    query_bank = to_typed_bank(query_spectra)

    query_precursors = harmonized_feature_table_filtered["M/Z"].to_numpy()
    query_precursors_reshaped = np.asarray(
        query_precursors, dtype=np.float32, order="C"
    ).reshape(-1)

    scores, idx = flashier_entropy_search(
        query_bank,
        query_precursors_reshaped,
        reference_spectra_list,
        reference_precursors,
        ms1_ppm_tolerance=20,  # 00000000,
        ms2_da_tolerance=0.05,
        min_matched_peaks=3,
        return_argmax=True,
    )

    feature_ids = harmonized_feature_table_filtered["FEATURE_ID"].to_numpy(dtype=str)

    library_ids = reference_ids[idx]
    library_ids = np.where(idx == -1, -1, library_ids)
    library_ids = library_ids.astype(int)
    output = pd.DataFrame(
        {
            cons.FEATURE_ID_COLUMN: feature_ids,
            cons.LIBRARY_ID_COLUMN: library_ids,
            cons.METHOD_COLUMN: "spectral_entropy",
            cons.INCHIKEY_COLUMN: reference_inchis[idx],
            cons.SCORE_COLUMN: scores,
            cons.LIBRARY_PRECURSOR_MZ_COLUMN: reference_precursors[idx],
            cons.LIBRARY_MZS_COLUMN: reference_mzs[idx],
            cons.LIBRARY_INTENSITIES_COLUMN: reference_intensities[idx],
        }
    )

    os.makedirs(os.path.dirname(output_feature_table), exist_ok=True)
    output.to_parquet(output_feature_table, index=False, engine="pyarrow")

    # delete the output DataFrame to free up memory
    del output

print(
    f"Spectral entropy annotation skipped for {counter}/{len(input_feature_tables)} files as output already exists."
)

100%|██████████| 39/39 [01:32<00:00,  2.38s/it]

Spectral entropy annotation skipped for 0/39 files as output already exists.


## Cosine similarity

In [8]:
counter = 0
for input_feature_table, output_feature_table_cosine in tqdm(
    zip(input_feature_tables, output_feature_tables_cosine),
    total=len(input_feature_tables),
):
    if os.path.exists(output_feature_table_cosine):
        counter += 1
        continue

    harmonized_feature_table = pd.read_parquet(input_feature_table)
    harmonized_feature_table_spectra_available = harmonized_feature_table[
        harmonized_feature_table["MS/MS_ASSIGNED"] == True
    ]

    harmonized_feature_table_filtered = filter_spectrum_peaks_in_df(
        harmonized_feature_table_spectra_available,
        "MS/MS_INTENSITIES",
        "MS/MS_MZS",
        10.0,
        0.01,
    )

    query_spectra = harmonized_feature_table_filtered.apply(
        lambda x: np.column_stack((x["MS/MS_MZS"], x["MS/MS_INTENSITIES"])), axis=1
    ).to_numpy()

    query_spectra = [clean_spectrum(spectrum) for spectrum in query_spectra]
    query_spectra = np.array(query_spectra, dtype=object)
    query_bank = to_typed_bank(query_spectra)

    query_precursors = harmonized_feature_table_filtered["M/Z"].to_numpy()
    query_precursors_reshaped = np.asarray(
        query_precursors, dtype=np.float32, order="C"
    ).reshape(-1)

    scores, idx = flash_cosine_search(
        query_bank,
        query_precursors_reshaped,
        reference_spectra_list,
        reference_precursors,
        ms1_da_tolerance=1.0,
        ms1_ppm_tolerance=20,
        ms2_da_tolerance=0.1,
        min_matched_peaks=3,
        return_argmax=True,
    )

    feature_ids = harmonized_feature_table_filtered["FEATURE_ID"].to_numpy(dtype=str)

    library_ids = reference_ids[idx]
    library_ids = np.where(idx == -1, -1, library_ids)
    library_ids = library_ids.astype(int)
    output = pd.DataFrame(
        {
            cons.FEATURE_ID_COLUMN: feature_ids,
            cons.LIBRARY_ID_COLUMN: library_ids,
            cons.METHOD_COLUMN: "cosine_similarity",
            cons.INCHIKEY_COLUMN: reference_inchis[idx],
            cons.SCORE_COLUMN: scores,
            cons.LIBRARY_PRECURSOR_MZ_COLUMN: reference_precursors[idx],
            cons.LIBRARY_MZS_COLUMN: reference_mzs[idx],
            cons.LIBRARY_INTENSITIES_COLUMN: reference_intensities[idx],
        }
    )

    os.makedirs(os.path.dirname(output_feature_table_cosine), exist_ok=True)
    output.to_parquet(output_feature_table_cosine, index=False, engine="pyarrow")

    # delete the output DataFrame to free up memory
    del output

print(
    f"Cosine similarity annotation skipped for {counter}/{len(input_feature_tables)} files as output already exists."
)

100%|██████████| 39/39 [12:31<00:00, 19.28s/it]

Cosine similarity annotation skipped for 0/39 files as output already exists.


## Dreams similarity